# 02 — Build Revenue Leakage Gold Model

## Purpose

Build an invoice-level Gold model that identifies confirmed revenue leakage, outstanding receivables, collection risk, failed payment exposure, and recovered retry revenue.

The model combines validated Silver invoices and payment attempts with the Gold Customer 360 customer profile.

## Grain

One record per Silver invoice.

## Sources

- `workspace.revenue_leakage_silver.invoices`
- `workspace.revenue_leakage_silver.payments`
- `workspace.revenue_leakage_gold.customer_360`

## Target

- `workspace.revenue_leakage_gold.revenue_leakage`

## Business Outcomes

- Quantify confirmed past-due revenue leakage
- Identify open revenue at risk
- Measure failed and pending payment exposure
- Attribute recovered revenue to successful retries
- Segment leakage by customer, subscription, risk tier, and value tier
- Support invoice aging and collection-priority analysis

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

SILVER_INVOICES_TABLE = (
    "workspace.revenue_leakage_silver.invoices"
)

SILVER_PAYMENTS_TABLE = (
    "workspace.revenue_leakage_silver.payments"
)

GOLD_CUSTOMER_360_TABLE = (
    "workspace.revenue_leakage_gold.customer_360"
)

GOLD_REVENUE_LEAKAGE_TABLE = (
    "workspace.revenue_leakage_gold.revenue_leakage"
)

EXPECTED_INVOICE_COUNT = 26_699
EXPECTED_PAYMENT_COUNT = 29_972
EXPECTED_CUSTOMER_COUNT = 5_150


silver_invoices_df = spark.table(
    SILVER_INVOICES_TABLE
)

silver_payments_df = spark.table(
    SILVER_PAYMENTS_TABLE
)

customer_360_source_df = spark.table(
    GOLD_CUSTOMER_360_TABLE
)


invoice_count = silver_invoices_df.count()

distinct_invoice_count = (
    silver_invoices_df
    .select("invoice_id")
    .distinct()
    .count()
)

payment_count = silver_payments_df.count()

distinct_payment_count = (
    silver_payments_df
    .select("payment_id")
    .distinct()
    .count()
)

customer_count = customer_360_source_df.count()

distinct_customer_count = (
    customer_360_source_df
    .select("customer_id")
    .distinct()
    .count()
)


invoice_customer_reference_errors = (
    silver_invoices_df
    .select(
        "invoice_id",
        "customer_id",
    )
    .join(
        customer_360_source_df.select(
            "customer_id"
        ),
        on="customer_id",
        how="left_anti",
    )
    .count()
)

payment_invoice_reference_errors = (
    silver_payments_df
    .select(
        "payment_id",
        "invoice_id",
    )
    .join(
        silver_invoices_df.select(
            "invoice_id"
        ),
        on="invoice_id",
        how="left_anti",
    )
    .count()
)


assert invoice_count == EXPECTED_INVOICE_COUNT, (
    "Unexpected Silver invoice count."
)

assert distinct_invoice_count == EXPECTED_INVOICE_COUNT, (
    "Duplicate Silver invoice IDs detected."
)

assert payment_count == EXPECTED_PAYMENT_COUNT, (
    "Unexpected Silver payment count."
)

assert distinct_payment_count == EXPECTED_PAYMENT_COUNT, (
    "Duplicate Silver payment IDs detected."
)

assert customer_count == EXPECTED_CUSTOMER_COUNT, (
    "Unexpected Customer 360 count."
)

assert distinct_customer_count == EXPECTED_CUSTOMER_COUNT, (
    "Duplicate Customer 360 IDs detected."
)

assert invoice_customer_reference_errors == 0, (
    "Silver invoices reference missing Gold customers."
)

assert payment_invoice_reference_errors == 0, (
    "Silver payments reference missing Silver invoices."
)


print(
    "Silver invoices: "
    f"{invoice_count:,}"
)

print(
    "Distinct invoice IDs: "
    f"{distinct_invoice_count:,}"
)

print(
    "Silver payment attempts: "
    f"{payment_count:,}"
)

print(
    "Distinct payment IDs: "
    f"{distinct_payment_count:,}"
)

print(
    "Customer 360 records: "
    f"{customer_count:,}"
)

print(
    "Invoice/customer reference errors: "
    f"{invoice_customer_reference_errors:,}"
)

print(
    "Payment/invoice reference errors: "
    f"{payment_invoice_reference_errors:,}"
)

print(
    "All required Revenue Leakage sources are available."
)

display(
    silver_invoices_df
    .groupBy("invoice_status")
    .agg(
        F.count("*").alias(
            "invoice_count"
        ),
        F.round(
            F.sum("invoice_total_amount"),
            2,
        ).alias(
            "invoice_total_amount"
        ),
        F.round(
            F.sum("amount_paid"),
            2,
        ).alias(
            "amount_paid"
        ),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias(
            "outstanding_amount"
        ),
        F.round(
            F.sum("voided_amount"),
            2,
        ).alias(
            "voided_amount"
        ),
    )
    .orderBy("invoice_status")
)

## 2. Build Invoice-Level Payment Metrics

Aggregate every validated Silver payment attempt to its invoice.

The resulting dataset measures successful, failed, pending, and retry attempts together with attempted, settled, failed, pending, and recovered amounts. The latest payment state is retained for leakage classification and collection prioritization.

In [0]:
latest_invoice_payment_window = (
    Window
    .partitionBy("invoice_id")
    .orderBy(
        F.col("payment_date").desc_nulls_last(),
        F.col("attempt_number").desc(),
        F.col("payment_id").desc(),
    )
)


latest_invoice_payment_df = (
    silver_payments_df
    .withColumn(
        "_latest_payment_rank",
        F.row_number().over(
            latest_invoice_payment_window
        ),
    )
    .filter(
        F.col("_latest_payment_rank") == 1
    )
    .select(
        "invoice_id",
        F.col("payment_id").alias(
            "latest_payment_id"
        ),
        F.col("provider_transaction_id").alias(
            "latest_provider_transaction_id"
        ),
        F.col("payment_status").alias(
            "latest_payment_status"
        ),
        F.col("attempt_number").alias(
            "latest_attempt_number"
        ),
        F.col("payment_date").alias(
            "latest_payment_date"
        ),
        F.col("settlement_date").alias(
            "latest_settlement_date"
        ),
        F.col("payment_method").alias(
            "latest_payment_method"
        ),
        F.col("payment_provider").alias(
            "latest_payment_provider"
        ),
        F.col("failure_reason").alias(
            "latest_failure_reason"
        ),
    )
)


invoice_payment_aggregates_df = (
    silver_payments_df
    .groupBy("invoice_id")
    .agg(
        F.count("*").alias(
            "payment_attempt_count"
        ),
        F.countDistinct("payment_id").alias(
            "distinct_payment_count"
        ),
        F.countDistinct(
            "provider_transaction_id"
        ).alias(
            "distinct_provider_transaction_count"
        ),
        F.sum(
            F.when(
                F.col("payment_status")
                == "Succeeded",
                1,
            ).otherwise(0)
        ).alias(
            "successful_payment_count"
        ),
        F.sum(
            F.when(
                F.col("payment_status")
                == "Failed",
                1,
            ).otherwise(0)
        ).alias(
            "failed_payment_count"
        ),
        F.sum(
            F.when(
                F.col("payment_status")
                == "Pending",
                1,
            ).otherwise(0)
        ).alias(
            "pending_payment_count"
        ),
        F.sum(
            F.when(
                F.col("attempt_number") > 1,
                1,
            ).otherwise(0)
        ).alias(
            "retry_attempt_count"
        ),
        F.sum(
            F.when(
                (
                    F.col("payment_status")
                    == "Succeeded"
                )
                & (
                    F.col("attempt_number") > 1
                ),
                1,
            ).otherwise(0)
        ).alias(
            "successful_retry_count"
        ),
        F.sum(
            F.when(
                (
                    F.col("payment_status")
                    == "Failed"
                )
                & (
                    F.col("attempt_number") > 1
                ),
                1,
            ).otherwise(0)
        ).alias(
            "failed_retry_count"
        ),
        F.sum(
            F.when(
                (
                    F.col("payment_status")
                    == "Succeeded"
                )
                & (
                    F.col("attempt_number") == 1
                ),
                1,
            ).otherwise(0)
        ).alias(
            "first_attempt_success_count"
        ),
        F.sum(
            F.when(
                (
                    F.col("payment_status")
                    == "Failed"
                )
                & (
                    F.col("attempt_number") == 1
                ),
                1,
            ).otherwise(0)
        ).alias(
            "first_attempt_failure_count"
        ),
        F.max("attempt_number").alias(
            "maximum_attempt_number"
        ),
        F.round(
            F.sum("transaction_amount"),
            2,
        ).alias(
            "payment_attempt_amount"
        ),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settled_amount"
        ),
        F.round(
            F.sum(
                F.when(
                    F.col("payment_status")
                    == "Failed",
                    F.col("transaction_amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "failed_payment_amount"
        ),
        F.round(
            F.sum(
                F.when(
                    F.col("payment_status")
                    == "Pending",
                    F.col("transaction_amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "pending_payment_amount"
        ),
        F.round(
            F.sum(
                F.when(
                    (
                        F.col("payment_status")
                        == "Succeeded"
                    )
                    & (
                        F.col("attempt_number") > 1
                    ),
                    F.col("settled_amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "recovered_retry_amount"
        ),
        F.array_sort(
            F.collect_set("payment_method")
        ).alias(
            "payment_methods"
        ),
        F.array_sort(
            F.collect_set("payment_provider")
        ).alias(
            "payment_providers"
        ),
        F.array_sort(
            F.collect_set("failure_reason")
        ).alias(
            "failure_reasons"
        ),
        F.min("payment_date").alias(
            "first_payment_date"
        ),
        F.max("payment_date").alias(
            "most_recent_payment_date"
        ),
    )
)


invoice_payment_metrics_df = (
    invoice_payment_aggregates_df
    .join(
        latest_invoice_payment_df,
        on="invoice_id",
        how="left",
    )
)

invoice_payment_metric_count = (
    invoice_payment_metrics_df.count()
)

distinct_invoice_payment_metric_count = (
    invoice_payment_metrics_df
    .select("invoice_id")
    .distinct()
    .count()
)

duplicate_invoice_payment_metric_count = (
    invoice_payment_metric_count
    - distinct_invoice_payment_metric_count
)

invoice_payment_reconciliation = (
    invoice_payment_metrics_df
    .agg(
        F.sum("payment_attempt_count").alias(
            "payment_attempt_count"
        ),
        F.round(
            F.sum("payment_attempt_amount"),
            2,
        ).alias(
            "payment_attempt_amount"
        ),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settled_amount"
        ),
        F.round(
            F.sum("recovered_retry_amount"),
            2,
        ).alias(
            "recovered_retry_amount"
        ),
    )
    .first()
)

assert duplicate_invoice_payment_metric_count == 0, (
    "Duplicate invoice payment metrics detected."
)

assert (
    invoice_payment_reconciliation[
        "payment_attempt_count"
    ]
    == EXPECTED_PAYMENT_COUNT
), (
    "Invoice-level payment attempts do not reconcile."
)

print(
    "Invoices with payment attempts: "
    f"{invoice_payment_metric_count:,}"
)

print(
    "Duplicate invoice payment metrics: "
    f"{duplicate_invoice_payment_metric_count:,}"
)

print(
    "Reconciled payment attempts: "
    f"{invoice_payment_reconciliation['payment_attempt_count']:,}"
)

print(
    "Payment attempt amount: "
    f"{invoice_payment_reconciliation['payment_attempt_amount']:,.2f}"
)

print(
    "Settled amount: "
    f"{invoice_payment_reconciliation['settled_amount']:,.2f}"
)

print(
    "Recovered retry amount: "
    f"{invoice_payment_reconciliation['recovered_retry_amount']:,.2f}"
)

display(
    invoice_payment_metrics_df
    .orderBy(
        F.desc("failed_payment_count"),
        F.desc("recovered_retry_amount"),
        "invoice_id",
    )
    .limit(20)
)

## 3. Assemble the Invoice-Level Revenue Leakage Dataset

Join every Silver invoice with its payment-attempt metrics and Customer 360 business context.

Classify confirmed leakage, open revenue at risk, recovered retry revenue, payment failures, invoice aging, and collection priority without double-counting invoice balances.

In [0]:
customer_leakage_context_df = (
    customer_360_source_df
    .select(
        "customer_id",
        "customer_name",
        "customer_status",
        "customer_segment",
        "country",
        "region",
        "risk_score",
        "risk_tier",
        "customer_value_tier",
        "current_mrr",
        "active_subscription_count",
        "analytics_snapshot_date",
    )
)


revenue_leakage_joined_df = (
    silver_invoices_df
    .join(
        invoice_payment_metrics_df,
        on="invoice_id",
        how="left",
    )
    .join(
        customer_leakage_context_df,
        on="customer_id",
        how="left",
    )
)


payment_numeric_columns = [
    "payment_attempt_count",
    "distinct_payment_count",
    "distinct_provider_transaction_count",
    "successful_payment_count",
    "failed_payment_count",
    "pending_payment_count",
    "retry_attempt_count",
    "successful_retry_count",
    "failed_retry_count",
    "first_attempt_success_count",
    "first_attempt_failure_count",
    "maximum_attempt_number",
    "payment_attempt_amount",
    "settled_amount",
    "failed_payment_amount",
    "pending_payment_amount",
    "recovered_retry_amount",
    "latest_attempt_number",
]

revenue_leakage_filled_df = (
    revenue_leakage_joined_df
    .fillna(
        0,
        subset=payment_numeric_columns,
    )
)


payment_array_columns = [
    "payment_methods",
    "payment_providers",
    "failure_reasons",
]

for column_name in payment_array_columns:
    revenue_leakage_filled_df = (
        revenue_leakage_filled_df
        .withColumn(
            column_name,
            F.when(
                F.col(column_name).isNull(),
                F.expr(
                    "CAST(array() AS ARRAY<STRING>)"
                ),
            ).otherwise(
                F.col(column_name)
            ),
        )
    )


revenue_leakage_enriched_df = (
    revenue_leakage_filled_df
    .withColumn(
        "invoice_age_days",
        F.datediff(
            F.col("analytics_snapshot_date"),
            F.col("invoice_date"),
        ),
    )
    .withColumn(
        "days_past_due",
        F.when(
            F.col("invoice_status") == "Past Due",
            F.greatest(
                F.datediff(
                    F.col("analytics_snapshot_date"),
                    F.col("due_date"),
                ),
                F.lit(0),
            ),
        ).otherwise(0),
    )
    .withColumn(
        "confirmed_leakage_amount",
        F.when(
            F.col("invoice_status") == "Past Due",
            F.col("outstanding_amount"),
        ).otherwise(
            F.lit(0)
        ),
    )
    .withColumn(
        "revenue_at_risk_amount",
        F.when(
            F.col("invoice_status") == "Open",
            F.col("outstanding_amount"),
        ).otherwise(
            F.lit(0)
        ),
    )
    .withColumn(
        "total_revenue_exposure_amount",
        F.round(
            F.col("confirmed_leakage_amount")
            + F.col("revenue_at_risk_amount"),
            2,
        ),
    )
    .withColumn(
        "recovered_revenue_amount",
        F.round(
            F.col("recovered_retry_amount"),
            2,
        ),
    )
)


revenue_leakage_classified_df = (
    revenue_leakage_enriched_df
    .withColumn(
        "invoice_aging_bucket",
        F.when(
            F.col("invoice_status") == "Voided",
            "Voided",
        )
        .when(
            F.col("invoice_status") == "Paid",
            "Paid",
        )
        .when(
            (
                F.col("invoice_status") == "Past Due"
            )
            & (
                F.col("days_past_due") <= 30
            ),
            "1-30 Days Past Due",
        )
        .when(
            (
                F.col("invoice_status") == "Past Due"
            )
            & (
                F.col("days_past_due") <= 60
            ),
            "31-60 Days Past Due",
        )
        .when(
            (
                F.col("invoice_status") == "Past Due"
            )
            & (
                F.col("days_past_due") <= 90
            ),
            "61-90 Days Past Due",
        )
        .when(
            F.col("invoice_status") == "Past Due",
            "90+ Days Past Due",
        )
        .when(
            (
                F.col("invoice_status") == "Open"
            )
            & (
                F.col("due_date")
                < F.col("analytics_snapshot_date")
            ),
            "Overdue Open",
        )
        .when(
            F.col("invoice_status") == "Open",
            "Current Open",
        )
        .otherwise("Other"),
    )
    .withColumn(
        "leakage_category",
        F.when(
            F.col("invoice_status") == "Voided",
            "Voided Invoice",
        )
        .when(
            F.col("confirmed_leakage_amount") > 0,
            "Confirmed Past-Due Leakage",
        )
        .when(
            F.col("revenue_at_risk_amount") > 0,
            "Open Receivable Risk",
        )
        .when(
            F.col("recovered_revenue_amount") > 0,
            "Recovered Retry Revenue",
        )
        .when(
            F.col("failed_payment_count") > 0,
            "Failed Collection Attempt",
        )
        .when(
            F.col("pending_payment_count") > 0,
            "Pending Collection",
        )
        .otherwise("No Revenue Exposure"),
    )
    .withColumn(
        "leakage_status",
        F.when(
            F.col("invoice_status") == "Voided",
            "Excluded",
        )
        .when(
            F.col("confirmed_leakage_amount") > 0,
            "Confirmed",
        )
        .when(
            F.col("revenue_at_risk_amount") > 0,
            "At Risk",
        )
        .when(
            F.col("recovered_revenue_amount") > 0,
            "Recovered",
        )
        .when(
            (
                F.col("failed_payment_count") > 0
            )
            | (
                F.col("pending_payment_count") > 0
            ),
            "Monitoring",
        )
        .otherwise("No Exposure"),
    )
)


raw_collection_priority_score = (
    F.when(
        F.col("confirmed_leakage_amount") > 0,
        45,
    ).otherwise(0)
    +
    F.when(
        F.col("revenue_at_risk_amount") > 0,
        20,
    ).otherwise(0)
    +
    F.when(
        F.col("days_past_due") > 90,
        25,
    )
    .when(
        F.col("days_past_due") > 60,
        20,
    )
    .when(
        F.col("days_past_due") > 30,
        15,
    )
    .when(
        F.col("days_past_due") > 0,
        10,
    )
    .otherwise(0)
    +
    F.when(
        F.col("failed_retry_count") > 0,
        10,
    ).otherwise(0)
    +
    F.when(
        F.col("pending_payment_count") > 0,
        10,
    ).otherwise(0)
    +
    F.when(
        F.col("risk_tier") == "High",
        15,
    )
    .when(
        F.col("risk_tier") == "Medium",
        8,
    )
    .otherwise(0)
    +
    F.when(
        F.col("customer_value_tier")
        == "High Value",
        5,
    ).otherwise(0)
)


revenue_leakage_df = (
    revenue_leakage_classified_df
    .withColumn(
        "collection_priority_score",
        F.when(
            F.col(
                "total_revenue_exposure_amount"
            ) > 0,
            F.least(
                F.lit(100),
                raw_collection_priority_score,
            ),
        ).otherwise(0),
    )
    .withColumn(
        "collection_priority",
        F.when(
            F.col("collection_priority_score")
            >= 75,
            "Critical",
        )
        .when(
            F.col("collection_priority_score")
            >= 50,
            "High",
        )
        .when(
            F.col("collection_priority_score")
            >= 25,
            "Medium",
        )
        .when(
            F.col("collection_priority_score")
            > 0,
            "Low",
        )
        .otherwise("None"),
    )
    .withColumn(
        "_gold_generated_at",
        F.current_timestamp(),
    )
)


revenue_leakage_count = (
    revenue_leakage_df.count()
)

distinct_revenue_leakage_invoice_count = (
    revenue_leakage_df
    .select("invoice_id")
    .distinct()
    .count()
)

duplicate_revenue_leakage_invoice_count = (
    revenue_leakage_count
    - distinct_revenue_leakage_invoice_count
)

assert revenue_leakage_count == EXPECTED_INVOICE_COUNT, (
    "Unexpected Revenue Leakage row count."
)

assert duplicate_revenue_leakage_invoice_count == 0, (
    "Duplicate Revenue Leakage invoice records detected."
)

print(
    "Revenue Leakage records: "
    f"{revenue_leakage_count:,}"
)

print(
    "Distinct Revenue Leakage invoices: "
    f"{distinct_revenue_leakage_invoice_count:,}"
)

print(
    "Duplicate Revenue Leakage invoices: "
    f"{duplicate_revenue_leakage_invoice_count:,}"
)

display(
    revenue_leakage_df
    .groupBy(
        "leakage_status",
        "leakage_category",
    )
    .agg(
        F.count("*").alias(
            "invoice_count"
        ),
        F.round(
            F.sum(
                "confirmed_leakage_amount"
            ),
            2,
        ).alias(
            "confirmed_leakage_amount"
        ),
        F.round(
            F.sum(
                "revenue_at_risk_amount"
            ),
            2,
        ).alias(
            "revenue_at_risk_amount"
        ),
        F.round(
            F.sum(
                "recovered_revenue_amount"
            ),
            2,
        ).alias(
            "recovered_revenue_amount"
        ),
    )
    .orderBy(
        "leakage_status",
        "leakage_category",
    )
)

## 4. Validate and Reconcile Revenue Leakage

Validate invoice uniqueness, required fields, payment-attempt relationships, invoice balances, leakage classifications, aging, collection priority, and nonnegative financial metrics.

Invoice, payment, exposure, settlement, and recovery totals must reconcile exactly with their validated Silver sources. A deterministic record hash supports idempotent Gold Delta merges.

In [0]:
gold_hash_columns = [
    column_name
    for column_name in revenue_leakage_df.columns
    if column_name not in {
        "_gold_generated_at",
        "_gold_record_hash",
    }
]

revenue_leakage_df = (
    revenue_leakage_df
    .withColumn(
        "_gold_record_hash",
        F.sha2(
            F.to_json(
                F.struct(
                    *[
                        F.col(column_name)
                        for column_name
                        in gold_hash_columns
                    ]
                ),
                options={
                    "ignoreNullFields": "false"
                },
            ),
            256,
        ),
    )
)


critical_fields = [
    "invoice_id",
    "customer_id",
    "subscription_id",
    "invoice_status",
    "analytics_snapshot_date",
    "leakage_category",
    "leakage_status",
    "collection_priority",
    "_gold_record_hash",
]

null_critical_field_condition = F.lit(False)

for column_name in critical_fields:
    null_critical_field_condition = (
        null_critical_field_condition
        | F.col(column_name).isNull()
    )


invalid_payment_count_condition = (
    F.col("payment_attempt_count")
    != (
        F.col("successful_payment_count")
        + F.col("failed_payment_count")
        + F.col("pending_payment_count")
    )
)

invalid_retry_count_condition = (
    F.col("retry_attempt_count")
    != (
        F.col("successful_retry_count")
        + F.col("failed_retry_count")
    )
)

invalid_invoice_balance_condition = (
    F.abs(
        F.col("invoice_total_amount")
        - (
            F.col("amount_paid")
            + F.col("outstanding_amount")
            + F.col("voided_amount")
        )
    ) > 0.01
)

invalid_settlement_condition = (
    F.abs(
        F.col("settled_amount")
        - F.col("amount_paid")
    ) > 0.01
)

invalid_exposure_condition = (
    (
        F.abs(
            F.col("total_revenue_exposure_amount")
            - (
                F.col("confirmed_leakage_amount")
                + F.col("revenue_at_risk_amount")
            )
        ) > 0.01
    )
    |
    (
        (
            F.col("invoice_status") == "Past Due"
        )
        & (
            F.abs(
                F.col("confirmed_leakage_amount")
                - F.col("outstanding_amount")
            ) > 0.01
        )
    )
    |
    (
        (
            F.col("invoice_status") == "Open"
        )
        & (
            F.abs(
                F.col("revenue_at_risk_amount")
                - F.col("outstanding_amount")
            ) > 0.01
        )
    )
    |
    (
        F.col("invoice_status").isin(
            "Paid",
            "Voided",
        )
        & (
            F.abs(
                F.col(
                    "total_revenue_exposure_amount"
                )
            ) > 0.01
        )
    )
)

invalid_recovery_condition = (
    (
        F.col("recovered_revenue_amount")
        < 0
    )
    |
    (
        F.col("recovered_revenue_amount")
        > F.col("settled_amount")
    )
)

invalid_date_condition = (
    (F.col("invoice_age_days") < 0)
    |
    (F.col("days_past_due") < 0)
    |
    (
        F.col("due_date")
        < F.col("invoice_date")
    )
)

invalid_priority_score_condition = (
    (
        F.col("collection_priority_score")
        < 0
    )
    |
    (
        F.col("collection_priority_score")
        > 100
    )
)

valid_leakage_mapping_condition = (
    (
        (F.col("invoice_status") == "Voided")
        & (
            F.col("leakage_status")
            == "Excluded"
        )
        & (
            F.col("leakage_category")
            == "Voided Invoice"
        )
    )
    |
    (
        (
            F.col("confirmed_leakage_amount")
            > 0
        )
        & (
            F.col("leakage_status")
            == "Confirmed"
        )
        & (
            F.col("leakage_category")
            == "Confirmed Past-Due Leakage"
        )
    )
    |
    (
        (
            F.col("revenue_at_risk_amount")
            > 0
        )
        & (
            F.col("leakage_status")
            == "At Risk"
        )
        & (
            F.col("leakage_category")
            == "Open Receivable Risk"
        )
    )
    |
    (
        (
            F.col("confirmed_leakage_amount")
            == 0
        )
        & (
            F.col("revenue_at_risk_amount")
            == 0
        )
        & (
            F.col("recovered_revenue_amount")
            > 0
        )
        & (
            F.col("leakage_status")
            == "Recovered"
        )
        & (
            F.col("leakage_category")
            == "Recovered Retry Revenue"
        )
    )
    |
    (
        (
            F.col("confirmed_leakage_amount")
            == 0
        )
        & (
            F.col("revenue_at_risk_amount")
            == 0
        )
        & (
            F.col("recovered_revenue_amount")
            == 0
        )
        & (
            (
                F.col("failed_payment_count")
                > 0
            )
            |
            (
                F.col("pending_payment_count")
                > 0
            )
        )
        & (
            F.col("leakage_status")
            == "Monitoring"
        )
    )
    |
    (
        (
            F.col("invoice_status")
            != "Voided"
        )
        & (
            F.col(
                "total_revenue_exposure_amount"
            ) == 0
        )
        & (
            F.col("recovered_revenue_amount")
            == 0
        )
        & (
            F.col("failed_payment_count")
            == 0
        )
        & (
            F.col("pending_payment_count")
            == 0
        )
        & (
            F.col("leakage_status")
            == "No Exposure"
        )
        & (
            F.col("leakage_category")
            == "No Revenue Exposure"
        )
    )
)

invalid_leakage_mapping_condition = (
    ~valid_leakage_mapping_condition
)

valid_priority_mapping_condition = (
    (
        (
            F.col("collection_priority_score")
            >= 75
        )
        & (
            F.col("collection_priority")
            == "Critical"
        )
    )
    |
    (
        (
            F.col("collection_priority_score")
            .between(50, 74)
        )
        & (
            F.col("collection_priority")
            == "High"
        )
    )
    |
    (
        (
            F.col("collection_priority_score")
            .between(25, 49)
        )
        & (
            F.col("collection_priority")
            == "Medium"
        )
    )
    |
    (
        (
            F.col("collection_priority_score")
            .between(1, 24)
        )
        & (
            F.col("collection_priority")
            == "Low"
        )
    )
    |
    (
        (
            F.col("collection_priority_score")
            == 0
        )
        & (
            F.col("collection_priority")
            == "None"
        )
    )
)

invalid_priority_mapping_condition = (
    ~valid_priority_mapping_condition
)


nonnegative_metric_columns = [
    "invoice_age_days",
    "days_past_due",
    "invoice_total_amount",
    "amount_paid",
    "outstanding_amount",
    "voided_amount",
    "payment_attempt_count",
    "successful_payment_count",
    "failed_payment_count",
    "pending_payment_count",
    "retry_attempt_count",
    "successful_retry_count",
    "failed_retry_count",
    "payment_attempt_amount",
    "settled_amount",
    "failed_payment_amount",
    "pending_payment_amount",
    "recovered_retry_amount",
    "confirmed_leakage_amount",
    "revenue_at_risk_amount",
    "total_revenue_exposure_amount",
    "recovered_revenue_amount",
    "collection_priority_score",
]

invalid_negative_metric_condition = (
    F.lit(False)
)

for column_name in nonnegative_metric_columns:
    invalid_negative_metric_condition = (
        invalid_negative_metric_condition
        | (F.col(column_name) < 0)
    )


validation_conditions = {
    "null_critical_field_count":
        null_critical_field_condition,
    "invalid_payment_count":
        invalid_payment_count_condition,
    "invalid_retry_count":
        invalid_retry_count_condition,
    "invalid_invoice_balance_count":
        invalid_invoice_balance_condition,
    "invalid_settlement_count":
        invalid_settlement_condition,
    "invalid_exposure_count":
        invalid_exposure_condition,
    "invalid_recovery_count":
        invalid_recovery_condition,
    "invalid_date_count":
        invalid_date_condition,
    "invalid_priority_score_count":
        invalid_priority_score_condition,
    "invalid_leakage_mapping_count":
        invalid_leakage_mapping_condition,
    "invalid_priority_mapping_count":
        invalid_priority_mapping_condition,
    "invalid_negative_metric_count":
        invalid_negative_metric_condition,
}


revenue_leakage_validation_row = (
    revenue_leakage_df
    .agg(
        F.count("*").alias(
            "revenue_leakage_count"
        ),
        F.countDistinct("invoice_id").alias(
            "distinct_invoice_count"
        ),
        *[
            F.sum(
                F.when(
                    condition,
                    1,
                ).otherwise(0)
            ).alias(metric_name)
            for metric_name, condition
            in validation_conditions.items()
        ],
    )
    .first()
)

duplicate_invoice_count = (
    revenue_leakage_validation_row[
        "revenue_leakage_count"
    ]
    - revenue_leakage_validation_row[
        "distinct_invoice_count"
    ]
)


silver_invoice_totals = (
    silver_invoices_df
    .agg(
        F.round(
            F.sum("invoice_total_amount"),
            2,
        ).alias("invoice_total_amount"),
        F.round(
            F.sum("amount_paid"),
            2,
        ).alias("amount_paid"),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias("outstanding_amount"),
        F.round(
            F.sum("voided_amount"),
            2,
        ).alias("voided_amount"),
    )
    .first()
)

gold_invoice_totals = (
    revenue_leakage_df
    .agg(
        F.round(
            F.sum("invoice_total_amount"),
            2,
        ).alias("invoice_total_amount"),
        F.round(
            F.sum("amount_paid"),
            2,
        ).alias("amount_paid"),
        F.round(
            F.sum("outstanding_amount"),
            2,
        ).alias("outstanding_amount"),
        F.round(
            F.sum("voided_amount"),
            2,
        ).alias("voided_amount"),
        F.round(
            F.sum("total_revenue_exposure_amount"),
            2,
        ).alias(
            "total_revenue_exposure_amount"
        ),
    )
    .first()
)

silver_payment_totals = (
    silver_payments_df
    .agg(
        F.count("*").alias(
            "payment_attempt_count"
        ),
        F.round(
            F.sum("transaction_amount"),
            2,
        ).alias(
            "payment_attempt_amount"
        ),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settled_amount"
        ),
        F.round(
            F.sum(
                F.when(
                    (
                        F.col("payment_status")
                        == "Succeeded"
                    )
                    & (
                        F.col("attempt_number") > 1
                    ),
                    F.col("settled_amount"),
                ).otherwise(0)
            ),
            2,
        ).alias(
            "recovered_retry_amount"
        ),
    )
    .first()
)

gold_payment_totals = (
    revenue_leakage_df
    .agg(
        F.sum("payment_attempt_count").alias(
            "payment_attempt_count"
        ),
        F.round(
            F.sum("payment_attempt_amount"),
            2,
        ).alias(
            "payment_attempt_amount"
        ),
        F.round(
            F.sum("settled_amount"),
            2,
        ).alias(
            "settled_amount"
        ),
        F.round(
            F.sum("recovered_revenue_amount"),
            2,
        ).alias(
            "recovered_retry_amount"
        ),
    )
    .first()
)


reconciliation_pairs = {
    "invoice_total_amount": (
        silver_invoice_totals[
            "invoice_total_amount"
        ],
        gold_invoice_totals[
            "invoice_total_amount"
        ],
    ),
    "amount_paid": (
        silver_invoice_totals[
            "amount_paid"
        ],
        gold_invoice_totals[
            "amount_paid"
        ],
    ),
    "outstanding_amount": (
        silver_invoice_totals[
            "outstanding_amount"
        ],
        gold_invoice_totals[
            "outstanding_amount"
        ],
    ),
    "voided_amount": (
        silver_invoice_totals[
            "voided_amount"
        ],
        gold_invoice_totals[
            "voided_amount"
        ],
    ),
    "exposure_to_outstanding": (
        silver_invoice_totals[
            "outstanding_amount"
        ],
        gold_invoice_totals[
            "total_revenue_exposure_amount"
        ],
    ),
    "payment_attempt_count": (
        silver_payment_totals[
            "payment_attempt_count"
        ],
        gold_payment_totals[
            "payment_attempt_count"
        ],
    ),
    "payment_attempt_amount": (
        silver_payment_totals[
            "payment_attempt_amount"
        ],
        gold_payment_totals[
            "payment_attempt_amount"
        ],
    ),
    "settled_amount": (
        silver_payment_totals[
            "settled_amount"
        ],
        gold_payment_totals[
            "settled_amount"
        ],
    ),
    "recovered_retry_amount": (
        silver_payment_totals[
            "recovered_retry_amount"
        ],
        gold_payment_totals[
            "recovered_retry_amount"
        ],
    ),
}


assert (
    revenue_leakage_validation_row[
        "revenue_leakage_count"
    ]
    == EXPECTED_INVOICE_COUNT
), (
    "Unexpected Revenue Leakage record count."
)

assert duplicate_invoice_count == 0, (
    "Duplicate Revenue Leakage invoices detected."
)

for metric_name in validation_conditions:
    assert (
        revenue_leakage_validation_row[
            metric_name
        ]
        == 0
    ), (
        f"Validation failed: {metric_name}"
    )

for metric_name, (
    silver_value,
    gold_value,
) in reconciliation_pairs.items():
    silver_numeric_value = float(
        silver_value or 0
    )

    gold_numeric_value = float(
        gold_value or 0
    )

    difference = abs(
        silver_numeric_value
        - gold_numeric_value
    )

    assert difference <= 0.01, (
        f"Reconciliation failed: {metric_name}"
    )


print(
    "revenue_leakage_count: "
    f"{revenue_leakage_validation_row['revenue_leakage_count']:,}"
)

print(
    "distinct_invoice_count: "
    f"{revenue_leakage_validation_row['distinct_invoice_count']:,}"
)

for metric_name in validation_conditions:
    print(
        f"{metric_name}: "
        f"{revenue_leakage_validation_row[metric_name]:,}"
    )

print(
    "duplicate_invoice_count: "
    f"{duplicate_invoice_count:,}"
)

for metric_name, (
    silver_value,
    gold_value,
) in reconciliation_pairs.items():
    difference = abs(
        float(silver_value or 0)
        - float(gold_value or 0)
    )

    print(
        f"{metric_name}: "
        f"Silver={float(silver_value or 0):,.2f}, "
        f"Gold={float(gold_value or 0):,.2f}, "
        f"Difference={difference:,.2f}"
    )

display(
    revenue_leakage_df
    .groupBy(
        "collection_priority",
        "leakage_status",
    )
    .agg(
        F.count("*").alias(
            "invoice_count"
        ),
        F.round(
            F.sum(
                "total_revenue_exposure_amount"
            ),
            2,
        ).alias(
            "total_revenue_exposure_amount"
        ),
        F.round(
            F.sum(
                "recovered_revenue_amount"
            ),
            2,
        ).alias(
            "recovered_revenue_amount"
        ),
    )
    .orderBy(
        "collection_priority",
        "leakage_status",
    )
)

## 5. Persist the Revenue Leakage Gold Table

Persist one analytics-ready record per validated Silver invoice in a managed Delta table.

The initial execution creates the Gold table. Subsequent executions use the deterministic invoice record hash and Delta `MERGE` to insert new invoices, update changed leakage records, and remove invoices no longer present in the current Silver state.

In [0]:
spark.sql(
    "CREATE SCHEMA IF NOT EXISTS "
    "workspace.revenue_leakage_gold"
)

revenue_leakage_df.createOrReplaceTempView(
    "revenue_leakage_gold_updates"
)

if spark.catalog.tableExists(
    GOLD_REVENUE_LEAKAGE_TABLE
):
    spark.sql(
        f"""
        MERGE INTO {GOLD_REVENUE_LEAKAGE_TABLE} AS target
        USING revenue_leakage_gold_updates AS source
            ON target.invoice_id = source.invoice_id

        WHEN MATCHED
            AND target._gold_record_hash
                <> source._gold_record_hash
            THEN UPDATE SET *

        WHEN NOT MATCHED
            THEN INSERT *

        WHEN NOT MATCHED BY SOURCE
            THEN DELETE
        """
    )

    write_method = "Delta MERGE"

else:
    (
        revenue_leakage_df
        .write
        .format("delta")
        .mode("overwrite")
        .option(
            "overwriteSchema",
            "true",
        )
        .saveAsTable(
            GOLD_REVENUE_LEAKAGE_TABLE
        )
    )

    write_method = (
        "Initial Delta table creation"
    )


saved_revenue_leakage_df = spark.table(
    GOLD_REVENUE_LEAKAGE_TABLE
)

saved_revenue_leakage_count = (
    saved_revenue_leakage_df.count()
)

saved_distinct_invoice_count = (
    saved_revenue_leakage_df
    .select("invoice_id")
    .distinct()
    .count()
)

saved_duplicate_invoice_count = (
    saved_revenue_leakage_count
    - saved_distinct_invoice_count
)

source_saved_reconciliation = (
    revenue_leakage_df
    .agg(
        F.round(
            F.sum(
                "total_revenue_exposure_amount"
            ),
            2,
        ).alias(
            "source_exposure_amount"
        ),
        F.round(
            F.sum(
                "recovered_revenue_amount"
            ),
            2,
        ).alias(
            "source_recovered_amount"
        ),
    )
    .crossJoin(
        saved_revenue_leakage_df
        .agg(
            F.round(
                F.sum(
                    "total_revenue_exposure_amount"
                ),
                2,
            ).alias(
                "saved_exposure_amount"
            ),
            F.round(
                F.sum(
                    "recovered_revenue_amount"
                ),
                2,
            ).alias(
                "saved_recovered_amount"
            ),
        )
    )
    .first()
)

exposure_difference = abs(
    float(
        source_saved_reconciliation[
            "source_exposure_amount"
        ]
        or 0
    )
    - float(
        source_saved_reconciliation[
            "saved_exposure_amount"
        ]
        or 0
    )
)

recovered_difference = abs(
    float(
        source_saved_reconciliation[
            "source_recovered_amount"
        ]
        or 0
    )
    - float(
        source_saved_reconciliation[
            "saved_recovered_amount"
        ]
        or 0
    )
)

assert (
    saved_revenue_leakage_count
    == EXPECTED_INVOICE_COUNT
), (
    "Unexpected saved Revenue Leakage count."
)

assert (
    saved_distinct_invoice_count
    == EXPECTED_INVOICE_COUNT
), (
    "Unexpected saved distinct invoice count."
)

assert saved_duplicate_invoice_count == 0, (
    "Duplicate saved Revenue Leakage invoices detected."
)

assert exposure_difference <= 0.01, (
    "Saved exposure amount does not reconcile."
)

assert recovered_difference <= 0.01, (
    "Saved recovered amount does not reconcile."
)

print(
    "Write method: "
    f"{write_method}"
)

print(
    "Gold table: "
    f"{GOLD_REVENUE_LEAKAGE_TABLE}"
)

print(
    "Saved Revenue Leakage records: "
    f"{saved_revenue_leakage_count:,}"
)

print(
    "Saved distinct invoice IDs: "
    f"{saved_distinct_invoice_count:,}"
)

print(
    "Saved exposure difference: "
    f"{exposure_difference:,.2f}"
)

print(
    "Saved recovered difference: "
    f"{recovered_difference:,.2f}"
)

display(
    saved_revenue_leakage_df
    .groupBy(
        "leakage_status",
        "collection_priority",
    )
    .agg(
        F.count("*").alias(
            "invoice_count"
        ),
        F.round(
            F.sum(
                "confirmed_leakage_amount"
            ),
            2,
        ).alias(
            "confirmed_leakage_amount"
        ),
        F.round(
            F.sum(
                "revenue_at_risk_amount"
            ),
            2,
        ).alias(
            "revenue_at_risk_amount"
        ),
        F.round(
            F.sum(
                "recovered_revenue_amount"
            ),
            2,
        ).alias(
            "recovered_revenue_amount"
        ),
    )
    .orderBy(
        "leakage_status",
        "collection_priority",
    )
)

## 6. Validate Idempotent Revenue Leakage Reprocessing

Rerun the Revenue Leakage Delta merge using the same validated invoice dataset.

The unchanged rerun must preserve all invoice records and financial totals while producing zero inserts, updates, or deletes.

In [0]:
revenue_leakage_before_rerun = (
    spark.table(
        GOLD_REVENUE_LEAKAGE_TABLE
    )
    .agg(
        F.count("*").alias(
            "row_count"
        ),
        F.round(
            F.sum(
                "total_revenue_exposure_amount"
            ),
            2,
        ).alias(
            "exposure_amount"
        ),
        F.round(
            F.sum(
                "recovered_revenue_amount"
            ),
            2,
        ).alias(
            "recovered_amount"
        ),
    )
    .first()
)

revenue_leakage_df.createOrReplaceTempView(
    "revenue_leakage_gold_updates"
)

spark.sql(
    f"""
    MERGE INTO {GOLD_REVENUE_LEAKAGE_TABLE} AS target
    USING revenue_leakage_gold_updates AS source
        ON target.invoice_id = source.invoice_id

    WHEN MATCHED
        AND target._gold_record_hash
            <> source._gold_record_hash
        THEN UPDATE SET *

    WHEN NOT MATCHED
        THEN INSERT *

    WHEN NOT MATCHED BY SOURCE
        THEN DELETE
    """
)

revenue_leakage_after_rerun_df = (
    spark.table(
        GOLD_REVENUE_LEAKAGE_TABLE
    )
)

revenue_leakage_after_rerun = (
    revenue_leakage_after_rerun_df
    .agg(
        F.count("*").alias(
            "row_count"
        ),
        F.countDistinct("invoice_id").alias(
            "distinct_invoice_count"
        ),
        F.round(
            F.sum(
                "total_revenue_exposure_amount"
            ),
            2,
        ).alias(
            "exposure_amount"
        ),
        F.round(
            F.sum(
                "recovered_revenue_amount"
            ),
            2,
        ).alias(
            "recovered_amount"
        ),
    )
    .first()
)

duplicate_invoices_after_rerun = (
    revenue_leakage_after_rerun[
        "row_count"
    ]
    - revenue_leakage_after_rerun[
        "distinct_invoice_count"
    ]
)

latest_revenue_leakage_history_df = (
    spark.sql(
        f"""
        DESCRIBE HISTORY
        {GOLD_REVENUE_LEAKAGE_TABLE}
        """
    )
    .orderBy(
        F.desc("version")
    )
    .limit(1)
)

latest_history_row = (
    latest_revenue_leakage_history_df
    .first()
)

latest_operation_metrics = (
    latest_history_row[
        "operationMetrics"
    ]
    or {}
)

rows_inserted_during_rerun = int(
    latest_operation_metrics.get(
        "numTargetRowsInserted",
        0,
    )
)

rows_updated_during_rerun = int(
    latest_operation_metrics.get(
        "numTargetRowsUpdated",
        0,
    )
)

rows_deleted_during_rerun = int(
    latest_operation_metrics.get(
        "numTargetRowsDeleted",
        0,
    )
)

exposure_difference_after_rerun = abs(
    float(
        revenue_leakage_before_rerun[
            "exposure_amount"
        ]
        or 0
    )
    - float(
        revenue_leakage_after_rerun[
            "exposure_amount"
        ]
        or 0
    )
)

recovered_difference_after_rerun = abs(
    float(
        revenue_leakage_before_rerun[
            "recovered_amount"
        ]
        or 0
    )
    - float(
        revenue_leakage_after_rerun[
            "recovered_amount"
        ]
        or 0
    )
)

assert (
    revenue_leakage_before_rerun[
        "row_count"
    ]
    == EXPECTED_INVOICE_COUNT
), (
    "Unexpected row count before rerun."
)

assert (
    revenue_leakage_after_rerun[
        "row_count"
    ]
    == EXPECTED_INVOICE_COUNT
), (
    "Unexpected row count after rerun."
)

assert rows_inserted_during_rerun == 0, (
    "Revenue Leakage rerun inserted unexpected rows."
)

assert rows_updated_during_rerun == 0, (
    "Revenue Leakage rerun updated unexpected rows."
)

assert rows_deleted_during_rerun == 0, (
    "Revenue Leakage rerun deleted unexpected rows."
)

assert duplicate_invoices_after_rerun == 0, (
    "Duplicate invoices detected after rerun."
)

assert exposure_difference_after_rerun <= 0.01, (
    "Exposure amount changed during rerun."
)

assert recovered_difference_after_rerun <= 0.01, (
    "Recovered amount changed during rerun."
)

print(
    "Rows before rerun: "
    f"{revenue_leakage_before_rerun['row_count']:,}"
)

print(
    "Rows after rerun: "
    f"{revenue_leakage_after_rerun['row_count']:,}"
)

print(
    "Rows inserted during rerun: "
    f"{rows_inserted_during_rerun:,}"
)

print(
    "Rows updated during rerun: "
    f"{rows_updated_during_rerun:,}"
)

print(
    "Rows deleted during rerun: "
    f"{rows_deleted_during_rerun:,}"
)

print(
    "Duplicate invoices after rerun: "
    f"{duplicate_invoices_after_rerun:,}"
)

print(
    "Exposure difference after rerun: "
    f"{exposure_difference_after_rerun:,.2f}"
)

print(
    "Recovered difference after rerun: "
    f"{recovered_difference_after_rerun:,.2f}"
)

print(
    "Revenue Leakage Gold processing is idempotent."
)

display(
    latest_revenue_leakage_history_df
    .select(
        "version",
        "timestamp",
        "operation",
        "operationMetrics",
    )
)

## 7. Final Result

The Revenue Leakage Gold model was created successfully as an invoice-level analytics Delta table.

### Output

- **Gold table:** `workspace.revenue_leakage_gold.revenue_leakage`
- **Invoice records:** 26,699
- **Distinct invoice IDs:** 26,699
- **Confirmed leakage invoices:** 3,641
- **Open invoices at risk:** 641
- **Recovered retry invoices:** 3,282
- **Invoices with no exposure:** 18,631
- **Excluded voided invoices:** 504
- **Confirmed past-due leakage:** 646,033.53 USD
- **Open revenue at risk:** 83,252.48 USD
- **Total outstanding exposure:** 729,286.01 USD
- **Recovered retry revenue:** 583,858.90 USD
- **Settled revenue:** 3,806,266.54 USD
- **Payment attempts reconciled:** 29,972

### Quality Guarantees

- One record per validated Silver invoice
- Zero duplicate invoice records
- Zero null critical fields
- Valid payment and retry relationships
- Exact invoice-balance reconciliation
- Exact Silver-to-Gold payment reconciliation
- Deterministic leakage and collection-priority classifications
- Deterministic Gold record hashes
- Idempotent Delta merge processing
- Zero inserts, updates, deletes, or financial changes during unchanged reprocessing